# Global Daily Event Analysis: Marine Heatwave ID & Tracking using `MarEx`

## `MarEx` Processing Pipeline for Unstructured Datasets

1. **Morphological Pre-Processing**
    - Components smaller than `prefilter_min_cells` are dropped from each day
    - Binary closing then opening, as repeated sparse-matrix dilation over the neighbour graph, fills holes and
      removes features up to `R_fill` cells (neighbour hops) across
    - Gaps of up to `T_fill` days within an event are bridged in time
    - Objects smaller than `area_filter_absolute` cells are dropped

2. **Blob Identification**
    - Connected components are labelled on each day (cells sharing an edge, by union-find over the mesh's neighbour graph)
    - Area and centroid are computed for every object

3. **Temporal Tracking**
    - Objects on consecutive days are linked where they overlap by at least `overlap_threshold` of the smaller area
    - When objects merge, the merged object keeps the parents' IDs, and each cell goes to the parent of the nearest
      cell (`nn_partitioning=True`) or of the nearest centroid

4. **Graph Reduction & Finalisation**
    - The overlap graph is reduced with `scipy.sparse.csgraph.connected_components` to one global ID per event
    - Events are mapped into ID-time space, with area, centroid, start and end for each

Every parameter is described under [Tracking & Merge/Split](https://marex.readthedocs.io/en/latest/guide/tracking.html#parameters).

### N.B.: The following `dask` config may be necessary on particular systems
```python
dask.config.set({
    'distributed.comm.timeouts.connect': '120s',  # Increase from default
    'distributed.comm.timeouts.tcp': '240s',      # Double the connection timeout
    'distributed.comm.retry.count': 10,           # More retries before giving up
})
```

In [ ]:
from getpass import getuser
from pathlib import Path

import xarray as xr

import marEx
import marEx.helper as hpc

In [ ]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "unstructured"  # where 01_preprocess_extremes.ipynb wrote its output
temp_dir = scratch_dir / "marEx" / "examples" / "tmp"  # staging for compute_mode="streaming"
comparison_period = slice("2013-01-01", "2013-12-31")  # centroid-partitioning comparison (one year)

# One node: 8 workers x 8 threads x 12 GB = 64 cores, 96 GB for the workers. The merge loop runs in the client
# (this kernel), so it gets a larger share of the 128 GB here than in the detection notebooks.
n_workers, threads_per_worker, memory_limit = 8, 8, "12GB"

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

In [ ]:
# Space stays whole because the dilation matrix and the connected-component labelling span the whole mesh.
# Time is the axis to chunk, the reverse of the detection notebook.
ds = xr.open_zarr(out_dir / "extremes_shifting_hobday.zarr", chunks={"time": 5, "ncells": -1})
ds

## Identification, Tracking, & Merging

The settings are the native-mesh equivalents of the gridded example's, so the two event sets describe the same physical
scales:

| | Gridded (0.25°) | Native mesh |
|---|---|---|
| `R_fill` | 12 cells | 80 cells (neighbour hops), the same length scale |
| `area_filter_absolute` | 600 cells | 13,549 cells: the same area, 600 × 549.6 km² (mean 0.25° ocean cell) / 24.34 km² (mean native cell) |
| `prefilter_min_cells` | not used | 23 cells, about one 0.25° cell |
| Polar cap | masked above 85°N | not masked: the bias it corrects is a property of the lat/lon grid |
| `T_fill`, `overlap_threshold`, `nn_partitioning` | 4, 0.25, True | 4, 0.25, True |

`prefilter_min_cells` drops components smaller than 23 cells from each day before the morphology. On the native mesh,
specks below the 0.25° scale act as stepping stones under the closing and join objects that the gridded run keeps
apart, and `area_filter_absolute` runs too late to undo that because it is applied after the closing.

At five years of daily data one int32 field is 1826 × 14.9 M × 4 B ≈ 109 GB, so the run uses
`compute_mode="streaming"`, which stages the whole-field intermediates to zarr under `temp_dir` (about 14 B per
cell-timestep, ≈ 380 GB here) instead of holding them in cluster memory. The staging directory outlives `run()`, so
the result is written first and the directory then released with `marEx.clear_staging`. See
[Unstructured Grid Tracking](https://marex.readthedocs.io/en/latest/guide/tracking.html#unstructured-meshes).

In [ ]:
with monitor.stage("Track: merging & splitting"):
    tracker = marEx.tracker(
        ds.extreme_events,
        ds.mask,
        R_fill=80,  # fill holes up to 80 neighbour hops in radius
        area_filter_absolute=13549,  # drop objects smaller than 13,549 cells (the gridded run's 600 cells, by area)
        prefilter_min_cells=23,  # drop specks below one 0.25° cell before the morphology
        T_fill=4,  # bridge gaps of up to 4 days within one event
        allow_merging=True,  # follow events through mergers and splits
        overlap_threshold=0.25,  # link objects that overlap by at least 25 % of the smaller area
        nn_partitioning=True,  # partition merged objects by nearest parent cell
        compute_mode="streaming",
        temp_dir=str(temp_dir),
        verbose=True,
        # Unstructured mesh
        unstructured_grid=True,
        dimensions={"x": "ncells"},
        coordinates={"x": "lon", "y": "lat"},
        neighbours=ds.neighbours,
        cell_areas=ds.cell_areas,
    )
    extreme_events_ds, merges_ds = tracker.run(return_merges=True)
    extreme_events_ds.to_zarr(out_dir / "events_merged.zarr", mode="w")
    merges_ds.to_netcdf(out_dir / "events_merged_merges.nc", mode="w")

marEx.clear_staging(extreme_events_ds)
extreme_events_ds = xr.open_zarr(out_dir / "events_merged.zarr", chunks={})
extreme_events_ds

In [ ]:
merges_ds

### Use Centroid-based Partitioning Method for Comparison

One year (`comparison_period`) with `nn_partitioning=False`, which assigns each cell of a merged object to the parent with the nearest centroid. Everything else is unchanged.

In [ ]:
with monitor.stage("Track: centroid partitioning (one year)"):
    tracker = marEx.tracker(
        ds.extreme_events.sel(time=comparison_period),
        ds.mask,
        R_fill=80,  # fill holes up to 80 neighbour hops in radius
        area_filter_absolute=13549,  # drop objects smaller than 13,549 cells (the gridded run's 600 cells, by area)
        prefilter_min_cells=23,  # drop specks below one 0.25° cell before the morphology
        T_fill=4,  # bridge gaps of up to 4 days within one event
        allow_merging=True,  # follow events through mergers and splits
        overlap_threshold=0.25,  # link objects that overlap by at least 25 % of the smaller area
        nn_partitioning=False,  # partition merged objects by nearest parent centroid (Sun et al. 2023)
        compute_mode="streaming",
        temp_dir=str(temp_dir),
        verbose=True,
        # Unstructured mesh
        unstructured_grid=True,
        dimensions={"x": "ncells"},
        coordinates={"x": "lon", "y": "lat"},
        neighbours=ds.neighbours,
        cell_areas=ds.cell_areas,
    )
    extreme_events_ds = tracker.run()
    extreme_events_ds.to_zarr(out_dir / "events_merged_centroid.zarr", mode="w")

marEx.clear_staging(extreme_events_ds)
extreme_events_ds = xr.open_zarr(out_dir / "events_merged_centroid.zarr", chunks={})
extreme_events_ds

## Wall Time and Memory

One row per stage, measured while the notebook ran on the cluster configured at the top. Peak worker memory is the
summed process memory the workers report to the scheduler, which is the quantity Dask compares against `memory_limit`.
The client (this kernel) is counted separately.

In [ ]:
monitor.summary()

In [ ]:
client.close()